# Pythonics — Writing Idiomatic Python

"Pythonic" code means writing Python the way the language was *intended* to be used — concise, readable, and expressive.

This appendix covers the following essential Pythonic concepts:

| Topic | What you'll learn |
|---|---|
| `*args` and `**kwargs` | Pass variable numbers of arguments to functions |
| `assert` | Write self-checking debugging guards |
| Context Managers | Safely manage resources like files and locks |
| Underscores | Understand naming conventions (`_var`, `__var`, etc.) |
| String Formatting | Four approaches from old-style `%` to modern f-strings |

---

# `*args` and `**kwargs`

Python functions normally accept a **fixed** number of arguments. But what if you don't know how many arguments will be passed?

Python solves this elegantly with two special syntaxes:

| Syntax | Name | Purpose |
|---|---|---|
| `*args` | Positional variadic | Collects extra **positional** arguments into a **tuple** |
| `**kwargs` | Keyword variadic | Collects extra **keyword** arguments into a **dict** |

> **Note:** The names `args` and `kwargs` are just conventions. What matters is the `*` and `**` prefix. You could write `*numbers` or `**options` — it would work the same way.

---

## `*args` — Variable-Length Positional Arguments

`*args` lets a function accept **any number of positional (non-keyword) arguments**.

Inside the function, `argv` is available as a **tuple** of all extra arguments passed in.

**When to use it:** When you want users to pass multiple values without wrapping them in a list.

In [ ]:
# f_arg: a regular required positional argument (always needed)
# *argv: collects ALL remaining positional arguments into a tuple called 'argv'
def test_var_args(f_arg, *argv):
    # Print the first required argument normally
    print("first normal arg:", f_arg)

    # Iterate over the tuple of extra arguments captured by *argv
    for arg in argv:
        print("another arg through *argv:", arg)

# Call the function with:
#   f_arg  = 'Dylan'           → the required positional argument
#   *argv  = ('green', 33, False, ['green', 'eggs', 'and', 'ham'])  → packed into a tuple
test_var_args('Dylan', 'green', 10 + 23, 3 < 2, ['green', 'eggs', 'and', 'ham'])

**What happened above?**
- `'Dylan'` → captured by `f_arg`
- `'green'`, `33`, `False`, `[...]` → all captured together in the `argv` **tuple**

Notice that `10 + 23` is evaluated to `33` and `3 < 2` is evaluated to `False` **before** being passed in. Python evaluates expressions before handing them to functions.

---

## `**kwargs` — Variable-Length Keyword Arguments

`**kwargs` lets a function accept **any number of keyword (named) arguments**.

Inside the function, `kwargs` is available as a **dictionary** where keys are argument names and values are the passed values.

**When to use it:** When you want named flexibility — e.g., configuration options, metadata, or named properties.

In [ ]:
# **kwargs captures all keyword arguments into a dictionary
# Each keyword=value pair in the call becomes a key-value pair in the dict
def greet_me(**kwargs):
    # .items() gives us (key, value) pairs from the kwargs dictionary
    for key, value in kwargs.items():
        print(key, ":", value)

# Each named argument (vocals=, guitar=, etc.) becomes a key in the kwargs dict
# Result: kwargs = {'vocals': 'Bono', 'guitar': 'The Edge', 'drums': 'Larry', 'bass': 'Adam'}
greet_me(vocals="Bono", guitar='The Edge', drums='Larry', bass='Adam')

**What happened above?**
- All keyword arguments are packed into a dictionary: `{'vocals': 'Bono', 'guitar': 'The Edge', ...}`
- We can then loop over them with `.items()` just like any dictionary

---

## `*args` vs `**kwargs` — Unpacking into a Function Call

The `*` and `**` operators work **both ways**:
- When **defining** a function: they **pack** arguments in
- When **calling** a function: they **unpack** a collection and spread it as arguments

This is useful when you have data stored in a tuple or dict and want to spread it as individual arguments.

In [ ]:
# A regular function with three fixed positional parameters
def test_args_kwargs(arg1, arg2, arg3):
    print("arg1:", arg1)
    print("arg2:", arg2)
    print("arg3:", arg3)

# --- APPROACH 1: Unpacking a tuple with *args ---
# Store the three values in a tuple
args = (5, "two", 3)

# The *args syntax 'unpacks' the tuple: equivalent to test_args_kwargs(5, "two", 3)
test_args_kwargs(*args)

In [ ]:
# --- APPROACH 2: Unpacking a dict with **kwargs (using {} literal) ---
# The keys of the dict MUST match the function parameter names exactly
kwargs = {"arg3": 3, "arg2": "two", "arg1": 5}

# Order doesn't matter when using **kwargs — Python matches by name
# Equivalent to: test_args_kwargs(arg1=5, arg2="two", arg3=3)
test_args_kwargs(**kwargs)

In [ ]:
# --- APPROACH 3: Unpacking a dict with **kwargs (using dict() constructor) ---
# dict() lets you create a dict with keyword syntax (no quotes needed for keys)
kwargs = dict(arg3=3, arg2="two", arg1=5)

# Functionally identical to Approach 2 — just a different way to build the dict
test_args_kwargs(**kwargs)

**Summary of `*args` / `**kwargs`:**

```
In function DEFINITION:  *collects* positional args into a tuple / keyword args into a dict
In function CALL:        *unpacks* a tuple into positional args / **unpacks* a dict into keyword args
```

The same `*` symbol does opposite things depending on context — this is fundamental Python.

---

# `assert` — Built-in Debugging Guard

The `assert` statement is Python's built-in way to **verify assumptions** during development.

**Syntax:**
```python
assert <condition>, <optional_error_message>
```

**How it works:**
- If `condition` is **True** → nothing happens, execution continues normally
- If `condition` is **False** → raises an `AssertionError` and stops execution

Think of it as saying: *"I'm asserting this must be true. If it isn't, something has gone very wrong."*

**Important rules:**
- ✅ Use asserts for **programmer errors** (bugs, impossible states)
- ❌ Do NOT use asserts for **user input validation** — they can be globally disabled with `-O` flag
- ❌ Do NOT use asserts for **recoverable errors** — use exceptions (`raise ValueError(...)`) instead

In [ ]:
# A function that applies a discount to a product's price
def apply_discount(product, discount):
    # Calculate the discounted price; int() truncates any fractional cents
    price = int(product['price'] * (1.0 - discount))

    # ASSERT: sanity-check that the computed price makes business sense.
    # This guards against two impossible situations:
    #   - price < 0       → negative discount (we're paying the customer?)
    #   - price > original → discount > 100% (price went UP after discount?)
    # If either condition is violated, AssertionError is raised immediately.
    assert 0 <= price <= product['price']

    # If the assert passed, return the valid discounted price
    return price

# Sample product: IBM stock at 149.00 (stored as integer cents: 14900)
security = {'Symbol': 'IBM', 'price': 14900}

# ✅ Valid: 25% discount → price = 11175 (between 0 and 14900)
print(apply_discount(security, 0.25))

# ✅ Valid: 100% discount → price = 0 (still between 0 and 14900)
print(apply_discount(security, 1.00))

# ❌ These would raise AssertionError — try uncommenting to see:
# apply_discount(security, -0.25)   # Negative discount → price exceeds original
# apply_discount(security, 1.5)     # 150% discount → negative price

**Key takeaways about `assert`:**

| | `assert` | `raise ValueError` |
|---|---|---|
| Purpose | Internal bugs / impossible states | Expected bad input from users |
| Can be disabled | Yes (`python -O script.py`) | No |
| Use in production? | No — strip with `-O` | Yes |

---

# Context Managers — Safe Resource Management

Resources like files, network connections, and locks must always be **released** after use — even if an error occurs midway.

Without care, exceptions can prevent cleanup code from running, causing **resource leaks**.

Python's `with` statement (context manager protocol) solves this elegantly by guaranteeing cleanup.

**How it works:**
1. Entering the `with` block calls `__enter__()` → acquires the resource
2. Exiting the block (normally or via exception) calls `__exit__()` → releases the resource

---

## The Problem: Unsafe File Handling

In [ ]:
# ❌ UNSAFE — Do NOT do this in production code

# Open the file and get a file descriptor
f = open('../Data/hello.txt', 'w')

# If this line throws an exception (e.g., disk full, encoding error),
# execution jumps to the error handler — skipping f.close() below!
f.write('hello, world')

# f.close() might NEVER be reached if write() throws an exception
# Result: file descriptor is leaked (OS resource not freed)
f.close()

## The Solution: Context Manager with `with`

In [ ]:
# ✅ SAFE — Always use this pattern for files

# 'with' opens the file and assigns it to 'f' via __enter__()
# When the 'with' block exits — for ANY reason (normal exit, exception, return) —
# Python automatically calls f.__exit__() which closes the file
with open('../Data/hello.txt', 'w') as f:
    # Even if this write() throws an exception, the file will still be closed!
    f.write('hello, world!')

# At this point, f is guaranteed to be closed — we don't need to call f.close()

## Context Managers for Thread Locks

In [ ]:
import threading

# Create a threading lock — a synchronization primitive
some_lock = threading.Lock()

# ❌ HARMFUL: Manual acquire/release pattern
# If an exception occurs in the 'try' block, the lock is still released
# BUT this is verbose and easy to get wrong
some_lock.acquire()      # Acquire the lock (blocks other threads)
try:
    # Critical section — only one thread runs this at a time
    print('Doing something')
finally:
    # Must be in 'finally' to ensure release even if exception occurs
    some_lock.release()  # Always release, even on exception

In [ ]:
# ✅ BETTER: Use 'with' — Lock implements the context manager protocol

# 'with some_lock' calls some_lock.acquire() at entry
# and some_lock.release() at exit — identical to try/finally but cleaner
with some_lock:
    # Critical section — automatically locked and unlocked
    print('Doing something')

# Lock is automatically released here, even if an exception occurred inside

**Context managers in other languages:**
- **C++**: RAII (Resource Acquisition Is Initialization)
- **Java**: `try-with-resources`
- **C#**: `using` statement

Python's `with` is the equivalent — the cleanest version among mainstream languages.

---

# The Meaning of Underscores in Python

Underscores in Python names carry significant meaning. Unlike many other conventions, some of these are **enforced by the interpreter**, not just custom.

| Pattern | Name | Enforced? | Meaning |
|---|---|---|---|
| `_var` | Single leading | Convention only | "Internal use" hint |
| `var_` | Single trailing | Convention only | Avoid keyword conflict |
| `__var` | Double leading | Interpreter enforced | Name mangling (private) |
| `__var__` | Double both sides | Convention only | Python magic / dunder |
| `_` | Single alone | Convention only | Throwaway / don't care |

---

## 1) Single Leading Underscore `_var` — Internal Use Hint

A leading underscore is a **gentle signal** to other developers: *"This is an internal implementation detail — don't use it directly from outside this class or module."*

Python does **not** enforce this — it's purely a convention. You can still access `_bar` from outside, but you're going against the author's intent.

In [ ]:
class Widget:
    def __init__(self):
        # Public attribute — intended to be freely accessed and modified from outside
        self.foo = 11

        # "Protected" attribute — the leading underscore signals "internal use only"
        # BUT Python does NOT actually prevent external access; it's just a convention
        self._bar = 23

w = Widget()

# Both are accessible — underscore is just a naming hint, not enforced
print(w.foo)   # ✅ Accessing public attribute — expected and fine
print(w._bar)  # ⚠️  Accessing "internal" attribute — works but goes against convention

## 2) Single Trailing Underscore `var_` — Avoiding Keyword Conflict

Python has reserved keywords (`class`, `def`, `lambda`, `filter`, `type`, etc.) that **cannot be used as variable names**.

When the ideal name for a variable happens to be a Python keyword, add a trailing underscore to sidestep the conflict.

In [ ]:
# ❌ This would cause a SyntaxError — 'class' is a reserved Python keyword
# def make_object(name, class):
#     pass

# ✅ Append a trailing underscore to resolve the naming conflict
# The trailing underscore makes it clear this is NOT the keyword 'class'
def make_object(name, class_):
    # class_ holds the class/category information without conflicting with the keyword
    pass

# Other common examples of trailing underscore usage:
# type_    (avoids conflict with built-in 'type')
# list_    (avoids conflict with built-in 'list')
# filter_  (avoids conflict with built-in 'filter')
# id_      (avoids conflict with built-in 'id')
print("Function defined successfully without syntax conflict")

## 3) Double Leading Underscore `__var` — Name Mangling

This is where underscores become truly interesting — and interpreter-enforced.

When Python sees `__var` inside a class, it **renames** it to `_ClassName__var`. This is called **name mangling**.

**Why?** To prevent accidental attribute name collisions when a class is subclassed.

Think of it as a privacy mechanism: `__var` in class `Foo` becomes `_Foo__var` — external code must use the mangled name to access it.

In [ ]:
class Gadget:
    def __init__(self):
        # Public attribute — stored exactly as 'foo'
        self.foo = 'Base Class foo'

        # Single underscore: 'protected' by convention only, stored as '_bar'
        self._bar = 'Base Class _bar'

        # Double underscore: MANGLED by Python interpreter
        # Internally stored as '_Gadget__baz' — not '__baz'!
        self.__baz = 'Base Class __baz'

g1 = Gadget()

# Access public and semi-private attributes directly
print(g1.foo)    # ✅ Works: stored as 'foo'
print(g1._bar)   # ✅ Works: stored as '_bar'

# g1.__baz would raise AttributeError! The attribute was mangled to '_Gadget__baz'
# Use dir() to see the actual attribute names in memory
print(dir(g1))   # Look for '_Gadget__baz' in the output — NOT '__baz'

In [ ]:
# Now let's see what happens when we subclass Gadget
class ExtendedGadget(Gadget):
    def __init__(self):
        # Call parent class __init__ to initialize parent's attributes
        super().__init__()

        # Override parent's 'foo' — just replaces the same attribute
        self.foo = 'Overridden foo'

        # Override parent's '_bar' — also just replaces the same attribute
        self._bar = 'Overridden _bar'

        # This creates a NEW mangled attribute: '_ExtendedGadget__baz'
        # It does NOT override '_Gadget__baz' from the parent!
        # Name mangling makes both coexist safely
        self.__baz = 'Overridden __baz'

g2 = ExtendedGadget()

# foo and _bar were genuinely overridden — only one exists
print(g2.foo)    # → 'Overridden foo'
print(g2._bar)   # → 'Overridden _bar'

# dir() reveals BOTH mangled attributes coexist:
#   '_Gadget__baz'         → parent's private attribute (still intact!)
#   '_ExtendedGadget__baz' → child's private attribute (new, separate)
print(dir(g2))

**Key insight:** Name mangling protects parent class internals from accidental overrides in child classes. The parent's `__baz` is *safely isolated* from the child's `__baz` because they get different mangled names.

---

## What is a "Dunder"?

**Dunder** = **D**ouble **under**score (pronounced as a word)

- `__baz` → pronounced **"dunder baz"**
- `__init__` → pronounced **"dunder init"** (not "dunder init dunder")
- `__call__` → pronounced **"dunder call"**

Dunders appear constantly throughout Python's standard library and OOP system.

---

## 4) Double Leading AND Trailing Underscore `__var__` — Magic Methods

Names surrounded by double underscores on **both sides** are Python's **special/magic/dunder methods**.

These are **not mangled** — they start and end with `__` so Python treats them differently.

They're reserved for **Python's own use** to implement core language behaviour:

| Dunder | Called when |
|---|---|
| `__init__` | Object is created |
| `__str__` | `str(obj)` or `print(obj)` |
| `__len__` | `len(obj)` |
| `__add__` | `obj + other` |
| `__call__` | `obj()` — makes object callable |

> **Best practice:** Don't invent your own `__myname__` attributes. Python may claim those names in future versions.

## 5) Single Underscore `_` — Throwaway Variable

A lone `_` is used as a **disposable variable name** — it signals *"I don't care about this value."*

Two common patterns:

In [ ]:
# Pattern 1: Loop counter you don't actually need
# '_' tells readers: "We're just iterating 32 times — the counter value is irrelevant"
for _ in range(32):
    print('Hello, World.')

In [ ]:
# Pattern 2: Tuple unpacking — ignore specific elements

# This function returns 4 values as a tuple
def make_gadget():
    # Returns: (colour, transmission, num_cylinders, mileage)
    return 'red', 'auto', 12, 3812.4

# We only care about 'colour' and 'mileage'
# Use '_' for the two middle values we want to throw away
# This is much cleaner than: colour, transmission, cylinders, mileage = make_gadget()
#                             (where transmission and cylinders are never used)
colour, _, _, mileage = make_gadget()

print(f"Colour: {colour}, Mileage: {mileage}")

---

# String Formatting — 4 Ways to Build Strings

Python has evolved through four distinct approaches to string formatting. Knowing all four helps you read older code and choose the right modern approach.

| Method | Introduced | Status | Use when |
|---|---|---|---|
| `%` operator | Python 2 | Legacy | Reading old code |
| `.format()` | Python 2.7 / 3.0 | Still valid | Need named substitutions |
| f-strings | Python 3.6 | **Preferred** | Almost always |
| Template strings | Python 2.4 | Niche | User-provided templates |

---

## 1) C-Style `%` Formatting — The Old Way

Borrowed directly from C's `printf` function. The `%` operator substitutes values into a format string using `%s` (string), `%d` (integer), `%f` (float) placeholders.

In [ ]:
# --- Single substitution ---

fav_song = "Hey Jude"

# %s is the placeholder for a string value
# The % operator replaces %s with the value on the right-hand side
s = 'Favourite song is %s' % fav_song

print(s)

In [ ]:
# --- Multiple substitutions ---

fname = "Bob"
lname = "Dylan"

# For multiple substitutions, wrap the values in a tuple on the right side
# %s placeholders are filled left-to-right from the tuple
s = 'Favourite singer is %s %s' % (fname, lname)

print(s)

# ⚠️ Limitations of %:
# - Easy to mismatch placeholder count vs. argument count
# - No named substitutions (have to use exact order)
# - Hard to read with many substitutions

## 2) `.format()` — "New Style" Formatting

Introduced in Python 3, backported to Python 2.7. Replaces `%` with `{}` placeholders and calls `.format()` on the string.

Key improvements over `%`:
- Can use **named** placeholders `{name}`
- Can **reuse** the same argument multiple times
- Supports rich formatting specifications

In [ ]:
# --- Single substitution with .format() ---

fav_song = "Hey Jude"

# {} is the anonymous placeholder — replaced by the first .format() argument
s = 'Favourite song is {}'.format(fav_song)

print(s)

In [ ]:
fname = "Bob"
lname = "Dylan"

# --- Anonymous placeholders (positional) ---
# {} placeholders are filled in order from left to right
s = 'Favourite singer is {} {}'.format(fname, lname)
print(s)

# --- Named placeholders ---
# {s1} and {s2} refer to keyword arguments passed to .format()
# Order of {} in the string doesn't need to match argument order
s = 'Favourite singer is {s1} {s2}'.format(s1=fname, s2=lname)
print(s)

## 3) f-Strings — Formatted String Literals (⭐ Preferred Modern Approach)

Introduced in Python 3.6, f-strings are the **recommended** way to format strings today.

**How they work:** Prefix the string with `f` (or `F`), then put any valid Python expression inside `{}`.

**Why f-strings win:**
- ✅ Fastest of all four methods
- ✅ Most readable — variables appear right in the string
- ✅ Full Python expressions allowed inside `{}`
- ✅ No mismatch between placeholders and variables

In [ ]:
# --- Basic variable substitution ---

fav_song = "Hey Jude"

# The 'f' prefix makes this an f-string
# {fav_song} is replaced at runtime with the variable's current value
s = f'Favourite song is, {fav_song}!'

print(s)

In [ ]:
a = 5
b = 10

# f-strings evaluate FULL Python expressions inside {}
# {a + b}       → evaluates addition: 15
# {2 * (a + b)} → evaluates multiplication: 30
# No need for intermediate variables — compute inline!
s = f'Five plus ten is {a + b} and not {2 * (a + b)}.'

print(s)

# More examples of what f-strings can embed:
# f'{name.upper()}'       → method calls
# f'{price:.2f}'          → format specifiers (2 decimal places)
# f'{"yes" if x > 0 else "no"}' → conditional expressions
# f'{len(my_list)}'       → function calls

## 4) Template Strings — Safest for User-Provided Templates

Template strings from the `string` module are the **simplest and least powerful** option.

**When to use them:** When the template string comes from an **untrusted source** (e.g., user input, config file). They don't evaluate arbitrary expressions, so they're safer.

Syntax: `$var` or `${var}` as placeholders.

In [ ]:
# Import Template from the standard library's 'string' module
from string import Template

# Create a Template object with $-prefixed placeholder variables
# $s1 and $s2 are the named substitution slots
t = Template('Favourite singer is $s1 $s2')

# .substitute() fills in the template with the provided keyword arguments
# s1=fname → replaces $s1 with fname's value
# s2=lname → replaces $s2 with lname's value
s = t.substitute(s1=fname, s2=lname)

print(s)

# Note: Template strings do NOT support arbitrary expressions like f-strings do.
# t = Template('Result: ${a + b}') would NOT work — Templates are deliberately limited.

---

## String Formatting — Quick Reference

```python
name = "Alice"
score = 95.5

# C-style (legacy)
'Name: %s, Score: %.1f' % (name, score)

# .format() (compatible)
'Name: {}, Score: {:.1f}'.format(name, score)

# f-string (preferred — Python 3.6+)
f'Name: {name}, Score: {score:.1f}'

# Template (safe for user input)
from string import Template
Template('Name: $n').substitute(n=name)
```

**Use f-strings by default.** Switch to Template only when the template comes from untrusted user input.